In [1]:
# Import core libraries
import os
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings('ignore')


In [2]:
class WiDSPreprocessor:
    def __init__(self, target_col='diabetes_mellitus', id_cols=None, 
                 missing_threshold=0.50, corr_threshold=0.95, var_threshold=0.01, smoothing=10.0):
        self.target_col = target_col
        self.id_cols = id_cols if id_cols is not None else ['encounter_id', 'hospital_id', 'Unnamed: 0', 'icu_id']
        self.missing_threshold = missing_threshold
        self.corr_threshold = corr_threshold
        self.var_threshold = var_threshold
        self.smoothing = smoothing
        
        # Stateful parameters to be fit on train split
        self.cols_to_drop_missing = []
        self.cat_cols = []
        self.num_cols = []
        self.low_card_cols = []
        self.high_card_cols = []
        
        self.cat_mappings = {}  # Safe label encoder mapping dict
        self.cat_imputer = None
        self.num_imputer = None
        self.capping_limits = {}
        self.ohe_encoder = None
        self.te_mappings = {}
        self.global_target_mean = None
        
        self.skewed_cols = []
        self.scaler = None
        self.cols_to_drop_low_var = []
        self.cols_to_drop_collinear = []
        self.final_features = []

    def fit(self, X, y):
        # 1. Segregate features
        X_temp = X.drop(columns=self.id_cols, errors='ignore')
        self.cat_cols = X_temp.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
        self.num_cols = X_temp.select_dtypes(include=[np.number]).columns.tolist()
        
        # Drop columns with too many missing values in train (>50%)
        missing_ratios = X_temp.isnull().mean()
        self.cols_to_drop_missing = missing_ratios[missing_ratios > self.missing_threshold].index.tolist()
        
        # Remove dropped columns from numerical/categorical working lists
        self.cat_cols = [c for c in self.cat_cols if c not in self.cols_to_drop_missing]
        self.num_cols = [c for c in self.num_cols if c not in self.cols_to_drop_missing]
        
        X_filtered = X.drop(columns=self.cols_to_drop_missing, errors='ignore')
        
        # Fit safe label encoder mapping (ignores NaNs)
        self.cat_mappings = {}
        for col in self.cat_cols:
            unique_vals = X_filtered[col].dropna().unique()
            self.cat_mappings[col] = {val: float(i) for i, val in enumerate(unique_vals)}
            
        # Map training categories to integers (preserving NaNs)
        X_cat_mapped = pd.DataFrame(index=X_filtered.index)
        for col in self.cat_cols:
            X_cat_mapped[col] = X_filtered[col].map(self.cat_mappings[col])
            
        # Fit Mode Imputer for mapped categoricals
        if self.cat_cols:
            self.cat_imputer = SimpleImputer(strategy='most_frequent')
            self.cat_imputer.fit(X_cat_mapped[self.cat_cols])
            
        # Fit MICE Imputer for numerical features
        if self.num_cols:
            self.num_imputer = IterativeImputer(max_iter=10, random_state=42, n_nearest_features=10)
            self.num_imputer.fit(X_filtered[self.num_cols])
            
        # Impute temporarily to compute limits, engineering, and encoding
        X_imputed = self._impute(X)
        
        # Fit Capping Limits (Winsorization at 1st & 99th percentiles)
        self.capping_limits = {}
        for col in self.num_cols:
            if X_imputed[col].nunique() > 2:
                q_low = X_imputed[col].quantile(0.01)
                q_high = X_imputed[col].quantile(0.99)
                self.capping_limits[col] = (q_low, q_high)
                
        X_capped = self._cap_outliers(X_imputed)
        
        # Feature Engineering (Clinical Interactions + aggregations)
        X_fe = self._engineer_features(X_capped, X_raw=X)
        
        # Update numeric columns list with engineered columns
        engineered_cols = [c for c in X_fe.columns if c not in X_capped.columns]
        curr_num_cols = self.num_cols + [c for c in engineered_cols if c not in self.id_cols]
        
        # Segregate low and high cardinality categoricals
        self.low_card_cols = []
        self.high_card_cols = []
        for col in self.cat_cols:
            if X_fe[col].nunique() <= 10:
                self.low_card_cols.append(col)
            else:
                self.high_card_cols.append(col)
                
        # Fit One-Hot Encoder for low cardinality
        if self.low_card_cols:
            self.ohe_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            self.ohe_encoder.fit(X_fe[self.low_card_cols])
            
        # Fit Target Encoding mappings with smoothing for high cardinality
        self.global_target_mean = y.mean()
        self.te_mappings = {}
        cols_to_te = self.high_card_cols + [c for c in ['hospital_id', 'icu_id'] if c in X.columns]
        for col in cols_to_te:
            if col in ['hospital_id', 'icu_id']:
                series = X[col]
            else:
                series = X_fe[col]
            stats = pd.DataFrame({'feat': series, 'target': y}).groupby('feat')['target'].agg(['count', 'mean'])
            smoothed = (stats['count'] * stats['mean'] + self.smoothing * self.global_target_mean) / (stats['count'] + self.smoothing)
            self.te_mappings[col] = smoothed.to_dict()
            
        # Transform temporarily to check skewness & scale
        X_encoded = self._encode(X_fe, X_raw=X)
        
        # Skewness detection (skewness > 1.5)
        self.skewed_cols = []
        for col in curr_num_cols:
            if col in X_encoded.columns and X_encoded[col].nunique() > 2:
                skew = X_encoded[col].skew()
                if abs(skew) > 1.5:
                    self.skewed_cols.append(col)
                    
        X_transformed = X_encoded.copy()
        for col in self.skewed_cols:
            X_transformed[col] = np.log1p(np.maximum(X_transformed[col], 0))
            
        # Fit RobustScaler
        self.scaler = RobustScaler()
        self.scaler.fit(X_transformed.drop(columns=self.id_cols, errors='ignore'))
        
        X_scaled = pd.DataFrame(
            self.scaler.transform(X_transformed.drop(columns=self.id_cols, errors='ignore')),
            columns=X_transformed.drop(columns=self.id_cols, errors='ignore').columns,
            index=X_transformed.index
        )
        
        # Low Variance Filter (< 0.01)
        variances = X_scaled.var()
        self.cols_to_drop_low_var = variances[variances < self.var_threshold].index.tolist()
        X_var_filtered = X_scaled.drop(columns=self.cols_to_drop_low_var)
        
        # Collinearity Filter (> 0.95 correlation)
        corr_matrix = X_var_filtered.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.cols_to_drop_collinear = [column for column in upper_tri.columns if any(upper_tri[column] > self.corr_threshold)]
        
        X_selected = X_var_filtered.drop(columns=self.cols_to_drop_collinear)
        self.final_features = X_selected.columns.tolist()
        
        return self

    def transform(self, X):
        # Apply sequential preprocessing steps without target leakage
        X_imputed = self._impute(X)
        X_capped = self._cap_outliers(X_imputed)
        X_fe = self._engineer_features(X_capped, X_raw=X)
        X_encoded = self._encode(X_fe, X_raw=X)
        
        X_transformed = X_encoded.copy()
        for col in self.skewed_cols:
            if col in X_transformed.columns:
                X_transformed[col] = np.log1p(np.maximum(X_transformed[col], 0))
                
        X_features = X_transformed.drop(columns=self.id_cols, errors='ignore')
        X_scaled = pd.DataFrame(self.scaler.transform(X_features), columns=X_features.columns, index=X_transformed.index)
        X_selected = X_scaled[self.final_features]
        
        # Re-attach target and ID columns safely
        for col in self.id_cols:
            if col in X.columns:
                X_selected[col] = X[col]
                
        return X_selected

    def _impute(self, X):
        X_out = X.copy()
        X_out = X_out.drop(columns=self.cols_to_drop_missing, errors='ignore')
        
        # Categorical Imputation
        if self.cat_cols:
            X_cat_mapped = pd.DataFrame(index=X_out.index)
            for col in self.cat_cols:
                X_cat_mapped[col] = X_out[col].map(self.cat_mappings.get(col, {}))
            X_cat_imputed = pd.DataFrame(self.cat_imputer.transform(X_cat_mapped), columns=self.cat_cols, index=X_out.index)
            for col in self.cat_cols:
                X_out[col] = X_cat_imputed[col]
                
        # Numerical Imputation (MICE)
        if self.num_cols:
            X_num_imputed = pd.DataFrame(self.num_imputer.transform(X_out[self.num_cols]), columns=self.num_cols, index=X_out.index)
            for col in self.num_cols:
                X_out[col] = X_num_imputed[col]
                
        return X_out

    def _cap_outliers(self, X):
        X_out = X.copy()
        for col, (q_low, q_high) in self.capping_limits.items():
            if col in X_out.columns:
                X_out[col] = X_out[col].clip(q_low, q_high)
        return X_out

    def _engineer_features(self, X, X_raw):
        fe_df = X.copy()
        
        # A. Clinical Interactions
        if 'd1_glucose_max' in fe_df.columns and 'd1_glucose_min' in fe_df.columns:
            fe_df['glucose_range'] = fe_df['d1_glucose_max'] - fe_df['d1_glucose_min']
            fe_df['glucose_ratio'] = fe_df['d1_glucose_max'] / (fe_df['d1_glucose_min'] + 1e-5)
        if 'd1_bun_max' in fe_df.columns and 'd1_creatinine_max' in fe_df.columns:
            fe_df['bun_creatinine_ratio'] = fe_df['d1_bun_max'] / (fe_df['d1_creatinine_max'] + 1e-5)
        if 'd1_sodium_max' in fe_df.columns and 'd1_potassium_max' in fe_df.columns:
            fe_df['sodium_potassium_ratio'] = fe_df['d1_sodium_max'] / (fe_df['d1_potassium_max'] + 1e-5)
        if 'd1_sodium_max' in fe_df.columns and 'd1_hco3_min' in fe_df.columns:
            fe_df['anion_gap_est'] = fe_df['d1_sodium_max'] - fe_df['d1_hco3_min']
        if 'bmi' in fe_df.columns:
            fe_df['obese_class'] = (fe_df['bmi'] >= 30).astype(float)
            
        comorbidity_cols = ['aids', 'cirrhosis', 'hepatic_failure', 'immunosuppression', 'leukemia', 'lymphoma', 'solid_tumor_with_metastasis']
        existing_comorb = [c for c in comorbidity_cols if c in fe_df.columns]
        if existing_comorb:
            fe_df['total_comorbidities'] = fe_df[existing_comorb].sum(axis=1)
            
        # B. Aggregations (Missing Lab / Visit Intensity)
        lab_cols = [c for c in X_raw.columns if c.startswith('d1_') or c.endswith('_apache') or c.startswith('h1_')]
        fe_df['missing_lab_intensity'] = X_raw[lab_cols].isnull().sum(axis=1).astype(float)
        if 'pre_icu_los_days' in fe_df.columns:
            fe_df['visit_intensity_los'] = fe_df['pre_icu_los_days']
            
        # C. Targeted Polynomial Interaction Features
        if 'age' in fe_df.columns and 'bmi' in fe_df.columns:
            fe_df['poly_age_bmi'] = fe_df['age'] * fe_df['bmi']
        if 'age' in fe_df.columns and 'd1_glucose_max' in fe_df.columns:
            fe_df['poly_age_glucose'] = fe_df['age'] * fe_df['d1_glucose_max']
        if 'bmi' in fe_df.columns and 'd1_glucose_max' in fe_df.columns:
            fe_df['poly_bmi_glucose'] = fe_df['bmi'] * fe_df['d1_glucose_max']
            
        return fe_df

    def _encode(self, X_fe, X_raw):
        X_out = X_fe.copy()
        if self.low_card_cols and self.ohe_encoder:
            ohe_data = self.ohe_encoder.transform(X_fe[self.low_card_cols])
            ohe_cols = self.ohe_encoder.get_feature_names_out(self.low_card_cols)
            ohe_df = pd.DataFrame(ohe_data, columns=ohe_cols, index=X_fe.index)
            X_out = pd.concat([X_out.drop(columns=self.low_card_cols), ohe_df], axis=1)
            
        cols_to_te = self.high_card_cols + [c for c in ['hospital_id', 'icu_id'] if c in X_raw.columns]
        for col in cols_to_te:
            mapping = self.te_mappings.get(col, {})
            val_series = X_raw[col] if col in ['hospital_id', 'icu_id'] else X_fe[col]
            X_out[f'{col}_encoded'] = val_series.map(mapping).fillna(self.global_target_mean)
            
        X_out = X_out.drop(columns=self.cat_cols, errors='ignore')
        return X_out


In [3]:
print("Loading datasets...")
train_df = pd.read_csv('data/TrainingWiDS2021.csv')
test_df = pd.read_csv('data/UnlabeledWiDS2021.csv')

target_col = 'diabetes_mellitus'
id_cols = ['encounter_id', 'hospital_id', 'icu_id']

# Separate target
y_train_full = train_df[target_col]
X_train_full = train_df.drop(columns=[target_col, 'Unnamed: 0'], errors='ignore')
X_test_full = test_df.drop(columns=['Unnamed: 0'], errors='ignore')

# Perform split-safe partition to prevent validation leakage
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, stratify=y_train_full, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")


Loading datasets...
X_train shape: (104125, 179)
X_val shape:   (26032, 179)


In [4]:
preprocessor = WiDSPreprocessor(target_col=target_col, id_cols=id_cols)
print("Fitting pipeline on X_train...")
preprocessor.fit(X_train, y_train)
print("Fitting complete.")


Fitting pipeline on X_train...
Fitting complete.


In [5]:
print("Transforming data splits...")
train_processed = preprocessor.transform(X_train)
val_processed = preprocessor.transform(X_val)
test_processed = preprocessor.transform(X_test_full)

# Re-attach labels to processed training/validation splits
train_processed[target_col] = y_train
val_processed[target_col] = y_val

print(f"Processed Train shape: {train_processed.shape}")
print(f"Processed Val shape:   {val_processed.shape}")
print(f"Processed Test shape:  {test_processed.shape}")


Transforming data splits...
Processed Train shape: (104125, 110)
Processed Val shape:   (26032, 110)
Processed Test shape:  (10234, 109)


In [6]:
os.makedirs('data', exist_ok=True)

print("Saving processed files...")
train_processed.to_parquet('data/train_split_processed.parquet', index=False)
val_processed.to_parquet('data/val_split_processed.parquet', index=False)
test_processed.to_parquet('data/test_processed.parquet', index=False)

print("Notebook pipeline run complete. Saved files in data/ directory.")


Saving processed files...
Notebook pipeline run complete. Saved files in data/ directory.
